#### Environment & Directory Setup

In [1]:
import pandas as pd
import yfinance as yf
import os

os.makedirs('../data/processed', exist_ok=True)

#### Fetching Live April 2026 Market Data (KC=F)

In [4]:
ticker_symbol = "KC=F"
market_data = yf.download(ticker_symbol, start="2024-10-01", end="2026-04-03")

kc_prices = market_data[['Close']].copy()
kc_prices.columns = ['KC_Cents_lb']
kc_prices.index = pd.to_datetime(kc_prices.index).astype('datetime64[ns]')

kc_prices['FOB_Price_USD_kg'] = ((kc_prices['KC_Cents_lb'] + 50) * 2.20462) / 100

print(f"Market Sync Complete. Current FOB Djibouti Price: ${kc_prices['FOB_Price_USD_kg'].iloc[-1]:.2f}/kg")

[*********************100%***********************]  1 of 1 completed

Market Sync Complete. Current FOB Djibouti Price: $7.61/kg


In [5]:
kc_prices.head()

,KC_Cents_lb,FOB_Price_USD_kg
Date,,
2024-10-01,264.149994,6.925814
2024-10-02,256.500000,6.757160
2024-10-03,252.050003,6.659055
2024-10-04,257.350006,6.775900
2024-10-07,244.649994,6.495913


#### The "Cost-Buildup" & Shrinkage Engine

In [6]:
SHRINKAGE_FACTOR = 0.985  # 1.5% moisture/handling loss from Station to Port
LOGISTICS_KG = 0.18       # Trucking, warehouse, and Djibouti port fees
ADMIN_FEE_LOT = 150       # ECTA Grading, scanning, and bank permit fees
SURRENDER_RATE = 0.50     # NBE Directive FXD/04/2026 (50% Forex Surrender)

df_registry = pd.read_csv('../data/raw/romina_registry_v2.csv')
df_registry['Inspection_Date'] = pd.to_datetime(df_registry['Inspection_Date']).astype('datetime64[ns]')
df_registry = df_registry.sort_values('Inspection_Date')

df_linked = pd.merge_asof(
    df_registry, 
    kc_prices.sort_index(), 
    left_on='Inspection_Date', 
    right_index=True, 
    direction='backward'
)

# 1. Net Exportable Weight (After 1.5% Shrinkage)
df_linked['Total_kg'] = df_linked['Quantity_MT'] * 1000
df_linked['Exportable_kg'] = df_linked['Total_kg'] * SHRINKAGE_FACTOR

# 2. Revenue (Based on Exportable weight at FOB Price)
df_linked['Gross_Revenue_USD'] = df_linked['Exportable_kg'] * df_linked['FOB_Price_USD_kg']

# 3. Shrinkage Value Loss
df_linked['Shrinkage_Loss_USD'] = (df_linked['Total_kg'] - df_linked['Exportable_kg']) * df_linked['FOB_Price_USD_kg']

# 4. Net Revenue (Subtracting Logistics and Admin Fees)
df_linked['Total_Costs_USD'] = (df_linked['Exportable_kg'] * LOGISTICS_KG) + ADMIN_FEE_LOT
df_linked['Net_Revenue_USD'] = df_linked['Gross_Revenue_USD'] - df_linked['Total_Costs_USD']

# 5. Forex Retention
df_linked['Retained_Forex_USD'] = df_linked['Net_Revenue_USD'] * SURRENDER_RATE

print(f"Audit Complete: 1.5% Shrinkage applied. Mass lost: {(df_linked['Total_kg'].sum() - df_linked['Exportable_kg'].sum()):,.2f} kg")

Audit Complete: 1.5% Shrinkage applied. Mass lost: 113,750.25 kg


In [7]:
df_linked.head()

,Lot_ID,Station,Woreda,Region,Inspection_Date,Quantity_MT,Status,Defect_Type,Cup_Score,EUDR_Verified,KC_Cents_lb,FOB_Price_USD_kg,Total_kg,Exportable_kg,Gross_Revenue_USD,Shrinkage_Loss_USD,Total_Costs_USD,Net_Revenue_USD,Retained_Forex_USD
399,ROM-25-1DB57,Dogada Yemasira,Nansabo,Oromia,2024-10-02,18.77,Passed,NaN,84.15,False,256.500000,6.757160,18770.0,18488.45,124929.420349,1902.478482,3477.921,121451.499349,60725.749674
281,ROM-25-BF42B,Astatke Menafesha,Wonsho,Sidama,2024-10-03,19.40,Passed,NaN,83.87,True,252.050003,6.659055,19400.0,19109.00,127247.877739,1937.784940,3589.620,123658.257739,61829.128870
261,ROM-25-07A5F,Astatke Menafesha,Wonsho,Sidama,2024-10-04,19.27,Passed,NaN,86.42,True,257.350006,6.775900,19270.0,18980.95,128613.013497,1958.573810,3566.571,125046.442497,62523.221249
32,ROM-25-979C5,Abebe Jengelo,Hoko Girja,Sidama,2024-10-05,18.64,Passed,NaN,80.68,True,257.350006,6.775900,18640.0,18360.40,124408.228936,1894.541557,3454.872,120953.356936,60476.678468
1,ROM-25-F9562,Belete Tanga,Bona,Sidama,2024-10-10,18.83,Rejected,Under-dried,75.21,True,254.750000,6.718579,18830.0,18547.55,124613.188278,1897.662766,3488.559,121124.629278,60562.314639


#### Final Senior Officer Briefing

In [8]:
total_portfolio = df_linked['Gross_Revenue_USD'].sum()
blocked_gross = df_linked[df_linked['Status'] == 'Rejected']['Gross_Revenue_USD'].sum()
trapped_forex = df_linked[df_linked['Status'] == 'Rejected']['Retained_Forex_USD'].sum()
total_shrinkage = df_linked['Shrinkage_Loss_USD'].sum()

print(f"ROMINA PLC: RE-SCALED REVENUE AUDIT")
print(f"Total Portfolio Gross (FOB Djibouti): ${total_portfolio:,.2f}")
print(f"Total Value Lost to Shrinkage: ${total_shrinkage:,.2f}")
print(f"Gross Revenue Trapped (93 Rejected Lots): ${blocked_gross:,.2f}")
print(f"CRITICAL: Trapped FOREX for Imports: ${trapped_forex:,.2f}")

# Save to Processed
df_linked.to_csv('../data/processed/romina_market_linked_v2.csv', index=False)
print("\nData exported to '../data/processed/romina_market_linked_v2.csv'")

--- ROMINA PLC: RE-SCALED REVENUE AUDIT (PHASE 2) ---
Total Portfolio Gross (FOB Djibouti): $66,890,906.42
Total Value Lost to Shrinkage: $1,018,643.24
Gross Revenue Trapped (93 Rejected Lots): $15,716,758.08
CRITICAL: Trapped FOREX for Imports: $7,695,060.90

Data exported to '../data/processed/romina_market_linked_v2.csv'
